In [ ]:
"""
Author: Aaron Niecestro

Created:    October 19, 2026
Editted:    October 21 2026
Status:     Completed

Description

K-Means (Euclidean; spherical clusters)

Evaluated using:

- Silhouette Score
- Davies–Bouldin Index
- Cluster stability
- PCA/t‑SNE/UMAP visualization
"""

=========================================================
0. Imports & Configuration
=========================================================

In [ ]:
from pathlib import Path

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
sns.set_theme(style="whitegrid", context="talk")

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

In [ ]:
RANDOM_STATE = 42

=========================================================
1. Load Data
=========================================================

In [ ]:
file_path = Path(
    r"C:\Users\aniec\Desktop\ibm-ml-project\unsupervised-ml\00-data\clean-data\winequality-final.csv"
)

In [ ]:
output_dir = Path(
    r"C:\Users\aniec\Desktop\ibm-ml-project\unsupervised-ml\02-Clustering\02-clustering-results"
)

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
wqrw = pd.read_csv(file_path)

In [ ]:
# Clean column names.
wqrw.columns = wqrw.columns.str.strip()

In [ ]:
print("----- DATA SHAPE (MASTER) -----")
print(f"Rows: {wqrw.shape[0]:,}")
print(f"Columns: {wqrw.shape[1]}")

In [ ]:
print("\n----- FIRST FIVE ROWS -----")
print(wqrw.head())

=========================================================
2. Column Type Setup
=========================================================

In [ ]:
required_cols = [
    "fixed acidity",
    "volatile acidity",
    "citric acid",
    "residual sugar",
    "chlorides",
    "free sulfur dioxide",
    "total sulfur dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
    "quality",
    "type",
]

In [ ]:
missing_cols = [
    col for col in required_cols
    if col not in wqrw.columns
]

In [ ]:
if missing_cols:
    raise ValueError(
        f"Missing expected Wine Quality columns: {missing_cols}"
    )

In [ ]:
# Categorical variable.
categorical_cols = ["type"]

In [ ]:
# Numeric columns, including quality.
numeric_cols = wqrw.select_dtypes(
    include=["number"]
).columns.tolist()

In [ ]:
# Exclude quality from the clustering features.
target = "quality"

In [ ]:
feature_cols = [
    col for col in numeric_cols
    if col != target
]

In [ ]:
print("\n----- NUMERIC COLUMNS -----")
print(numeric_cols)

In [ ]:
print("\n----- CATEGORICAL COLUMNS -----")
print(categorical_cols)

In [ ]:
print("\n----- CLUSTERING FEATURES -----")
print(feature_cols)

In [ ]:
print("\n----- EXCLUDED INTERPRETATION VARIABLE -----")
print(target)

=========================================================
3. Feature Matrix for Clustering
=========================================================

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
X = wqrw[feature_cols].copy()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
print("Feature matrix shape:", X_scaled.shape)

=========================================================
4. K-Means Evaluation
=========================================================

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
)

In [ ]:
k_results = []

In [ ]:
k_range = range(2, 21)

In [ ]:
for k in k_range:

    kmeans = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=20,
        random_state=42,
    )

    labels = kmeans.fit_predict(X_scaled)

    silhouette = silhouette_score(
        X_scaled,
        labels
    )

    db_index = davies_bouldin_score(
        X_scaled,
        labels
    )

    k_results.append(
        {
            "k": k,
            "silhouette_score": silhouette,
            "davies_bouldin_index": db_index,
            "inertia": kmeans.inertia_,
        }
    )

=========================================================    
Results Table
=========================================================

In [ ]:
results_df = pd.DataFrame(k_results)

In [ ]:
print(results_df.round(4))

In [ ]:
results_df.to_csv(
    output_dir / "kmeans_evaluation.csv",
    index=False
)

> 0.90 = Very stable

=========================================================
5. Evaluation Plots
=========================================================

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5)
)

In [ ]:
# Inertia
axes[0].plot(
    results_df["k"],
    results_df["inertia"],
    marker="o"
)
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")

In [ ]:
# Silhouette
axes[1].plot(
    results_df["k"],
    results_df["silhouette_score"],
    marker="o"
)
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Score")

In [ ]:
# Davies-Bouldin
axes[2].plot(
    results_df["k"],
    results_df["davies_bouldin_index"],
    marker="o"
)
axes[2].set_title("Davies-Bouldin Index")
axes[2].set_xlabel("k")
axes[2].set_ylabel("Index")

In [ ]:
plt.tight_layout()
plt.show()

In [ ]:
"""
Interpretation

Higher Silhouette Score = better cluster separation.
Lower Davies-Bouldin Index = better cluster quality.
Elbow point in Inertia helps identify diminishing returns from increasing k.

- Silhouette Score (higher is better) Scale:

Near 1.0 = excellent separation
Around 0.5 = good
Around 0.25 = weak but usable
Near 0 = clusters overlap heavily

- Davies-Bouldin Index (lower is better)

- Elbow Method

The line always decreases because adding clusters always reduces inertia.
What you're looking for is: Huge improvements, not small improvements
"""

In [ ]:
"""
Insights:
    
**Based on Graph:**

Looking at all three plots together, I'd choose k = 4.

1. Silhouette Score

The highest score is:
k = 2 => ~0.27
But k=2 is often too simplistic for a dataset with many wine characteristics.
The next strong value is:
k = 4 => ~0.24
After k=4, the scores generally trend downward.

Favorable: k = 4

2. Davies-Bouldin Index

Lower is better.
The lowest point appears to be:
k = 4 => ~1.49
That's the best DB score across all tested k values.

Strongly favors: k = 4

3. Elbow Method

The biggest improvements occur between:
2 => 3
3 => 4
4 => 5
After about 4-5, the curve starts flattening and each additional cluster gives a smaller reduction in inertia.

Elbow around: 4-5

Although the highest Silhouette Score occurred at k=2, the Davies-Bouldin Index 
reached its minimum at k=4, indicating better cluster compactness and separation. 
The Elbow Method also showed diminishing reductions in inertia beyond four clusters. 
Therefore, k=4 was selected as the optimal clustering solution because it 
provided the best balance between performance metrics and interpretability.

Simple decision table
Metric          Best    Choosen
kSilhouette     2        4
Davies-Bouldin	4        4
Elbow           4-5      4-5
Overall Choice	4        4

For a business or portfolio presentation, k=4 is much easier to justify than k=2, 
because all three evaluation approaches are reasonably aligned around that value.

**Based on numbers:**

Strongest candidate: k = 4
k	Silhouette	DB Index	Verdict2	0.2704	1.6263	Best silhouette but very simple segmentation
3	0.2300	1.5074	Good
4	0.2428	1.4844	✅ Best overall
5+	Much lower	Mostly worse	Not competitive
Why not k = 2?

Technically, k=2 has the highest silhouette score.

However:

K-Means often favors fewer clusters on silhouette.
The improvement from 0.2428 to 0.2704 is not huge.
The DB Index is noticeably better at k=4.
Four clusters provide much richer business interpretation than just two wine groups.

I'd rather explain:

"Four distinct wine profiles were discovered" than
"The wines split into two groups."

Why not k > 4?

After k=4:

Silhouette drops sharply to 0.1769 at k=5.
Never recovers.
Davies-Bouldin never beats 1.4844 again.
Elbow curve starts flattening.

So you're adding complexity without improving clustering quality.

K-Means clustering was evaluated for k values ranging from 2 to 20. 
While the highest silhouette score was observed at k=2 (0.2704), 
the Davies-Bouldin Index reached its minimum value at k=4 (1.4844), 
indicating superior cluster compactness and separation. Additionally, 
the elbow plot showed diminishing reductions in inertia beyond four clusters. 
Therefore, k=4 was selected as the optimal clustering solution, balancing 
cluster quality, interpretability, and model complexity.

Choosing k = 4 and justifying it using:

Lowest Davies-Bouldin Index
Near-maximum Silhouette Score
Elbow around k=4
Better interpretability than k=2

So unless your PCA/t-SNE plots look extremely strange, I'd proceed with k=4 as 
the final K-Means model.
"""

=========================================================
6. Cluster Stability
=========================================================

In [ ]:
"""
A simple approach is to rerun K-Means with different random seeds and compare 
solutions using Adjusted Rand Index (ARI).
"""

In [ ]:
from sklearn.metrics import adjusted_rand_score

In [ ]:
best_k = 4  # replace with chosen k found above

In [ ]:
seeds = [1, 11, 21, 31, 41]

In [ ]:
cluster_labels = []

In [ ]:
for seed in seeds:

    km = KMeans(
        n_clusters=best_k,
        init="k-means++",
        n_init=20,
        random_state=seed
    )

    cluster_labels.append(
        km.fit_predict(X_scaled)
    )

In [ ]:
ari_scores = []

In [ ]:
for i in range(len(cluster_labels)-1):

    ari = adjusted_rand_score(
        cluster_labels[i],
        cluster_labels[i+1]
    )

    ari_scores.append(ari)

In [ ]:
print("ARI Scores:", ari_scores)
print("Average ARI:", np.mean(ari_scores))

In [ ]:
'''
Rule of thumb

ARI > 0.90 => very stable
ARI 0.70–0.90 => reasonably stable
ARI < 0.70 => unstable cluster structure
'''

=========================================================
7. Final K-Means Model
=========================================================

Fit final K-Means model using selected k.

In [ ]:
best_k = 4

In [ ]:
kmeans_final = KMeans(
    n_clusters=best_k,
    init="k-means++",
    n_init=20,
    random_state=RANDOM_STATE
)

In [ ]:
labels = kmeans_final.fit_predict(X_scaled)

Save cluster assignments to the dataframe.

In [ ]:
wqrw["kmeans_cluster"] = labels

=========================================================
8. PCA Visualization
=========================================================

Project data into two principal components.

In [ ]:
pca = PCA(n_components=2)

In [ ]:
X_pca = pca.fit_transform(X_scaled)

In [ ]:
plt.figure(figsize=(8, 6))

In [ ]:
scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=labels,
    cmap="tab10",
    alpha=0.7
)

In [ ]:
plt.title(f"K-Means Clusters (k={best_k}) - PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(scatter)

In [ ]:
plt.show()

=========================================================
9. t-SNE Visualization
=========================================================

Project data using t-SNE.

In [ ]:
from sklearn.manifold import TSNE

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=30,
    random_state=RANDOM_STATE
)

In [ ]:
X_tsne = tsne.fit_transform(X_scaled)

In [ ]:
plt.figure(figsize=(8, 6))

In [ ]:
plt.scatter(
    X_tsne[:, 0],
    X_tsne[:, 1],
    c=labels,
    cmap="tab10",
    alpha=0.7
)

In [ ]:
plt.title(f"K-Means Clusters (k={best_k}) - t-SNE")

In [ ]:
plt.show()

=========================================================
10. UMAP Visualization
=========================================================

Project data using UMAP.

In [ ]:
try:

    import umap.umap_ as umap

    reducer = umap.UMAP(
        n_components=2,
        random_state=RANDOM_STATE
    )

    X_umap = reducer.fit_transform(X_scaled)

    plt.figure(figsize=(8, 6))

    plt.scatter(
        X_umap[:, 0],
        X_umap[:, 1],
        c=labels,
        cmap="tab10",
        alpha=0.7
    )

    plt.title(f"K-Means Clusters (k={best_k}) - UMAP")

    plt.show()

In [ ]:
except ModuleNotFoundError:

    print("UMAP not installed. Skipping UMAP visualization.")

In [ ]:
'''
I would recommend selecting the final k based primarily on Silhouette Score, 
validating it with the Davies-Bouldin  Index, confirming robustness via ARI 
stability checks, and then presenting PCA, t-SNE, and UMAP cluster visualizations 
to support the interpretation.

PCA, t-SNE, and UMAP visualizations were used to evaluate cluster separation. 
While PCA showed moderate overlap between neighboring groups, both t-SNE and 
UMAP revealed four distinct cluster regions with relatively clear boundaries. 
The visualization results support the quantitative evaluation metrics and 
provide additional evidence that k=4 captures meaningful structure within 
the wine quality dataset.
'''

=========================================================
11. Cluster Profile Table
=========================================================

Calculate average feature values by cluster.

In [ ]:
cluster_profile = (
    wqrw
    .groupby("kmeans_cluster")
    [feature_cols + ["quality"]]
    .mean()
    .round(2)
)

In [ ]:
print("\n----- CLUSTER PROFILE -----")
print(cluster_profile)

In [ ]:
cluster_profile.to_csv(
    output_dir / "kmeans_cluster_profile.csv"
)

In [ ]:
"""
This lets you write findings such as:

Cluster 0 has highest alcohol
Cluster 1 has highest residual sugar
Cluster 2 has lowest pH
Cluster 3 has highest sulphates

That turns clustering from "pretty plots" into actual business insights.
"""

=========================================================
12. Cluster Size Distribution
=========================================================

Calculate cluster membership counts.

In [ ]:
cluster_sizes = (
    wqrw["kmeans_cluster"]
    .value_counts()
    .sort_index()
)

In [ ]:
print("\n----- CLUSTER SIZES -----")
print(cluster_sizes)

In [ ]:
cluster_sizes.to_csv(
    output_dir / "kmeans_cluster_sizes.csv"
)

In [ ]:
cluster_sizes.plot(
    kind="bar",
    figsize=(8, 5)
)

In [ ]:
plt.title("Cluster Sizes")
plt.xlabel("Cluster")
plt.ylabel("Count")

In [ ]:
plt.show()

=========================================================
13. Save Final Dataset
=========================================================

Save dataset with cluster assignments.

In [ ]:
wqrw.to_csv(
    output_dir / "winequality_kmeans_clusters.csv",
    index=False
)

=========================================================
Print cluster quality summary.
=========================================================

In [ ]:
print("\n----- FINAL K-MEANS SUMMARY -----")

In [ ]:
print(
    f"Selected k: {best_k}"
)

In [ ]:
print(
    f"Silhouette Score: "
    f"{results_df.loc[results_df['k'] == best_k, 'silhouette_score'].iloc[0]:.4f}"
)

In [ ]:
print(
    f"Davies-Bouldin Index: "
    f"{results_df.loc[results_df['k'] == best_k, 'davies_bouldin_index'].iloc[0]:.4f}"
)

In [ ]:
print(
    f"Average ARI: "
    f"{np.mean(ari_scores):.4f}"
)

In [ ]:
"""
This is actually a very strong K-Means result.

What stands out
1. Cluster stability is excellent
Average ARI: 0.9986

Rule of thumb: Average ARI > 0.90 = Very stable

My result: Average ARI = 0.9986

This is almost perfect stability.

That means changing the random seed barely changes the cluster assignments.

You can confidently write:

K-Means clustering exhibited exceptional stability across multiple random initializations, achieving an average Adjusted Rand Index (ARI) of 0.9986, indicating highly consistent cluster assignments.

2. Cluster sizes are reasonable

Cluster 0 = 854
Cluster 1 = 2473
Cluster 2 = 1406
Cluster 3 = 587

None are tiny.
None dominate the dataset.
That's typically what you want.

3. The profiles are actually interpretable

- Cluster 0

Alcohol: 10.23
Volatile Acidity: 0.61
Quality: 5.38

Characteristics:
higher volatile acidity
lower alcohol
lowest quality

Possible label: Traditional lower-quality acidic wines

- Cluster 1

Alcohol: 11.19
Quality: 6.07
Residual Sugar: 3.26

Characteristics:
highest quality
highest alcohol
relatively low acidity

Possible label: Higher-quality balanced wines

- Cluster 2

Residual Sugar: 10.72
Total SO2: 171.50
Alcohol: 9.62

Characteristics:
by far highest residual sugar
highest sulfur dioxide
lowest alcohol

This cluster jumps out immediately.

Possible label: Sweet / high-sugar wines

- Cluster 3

Fixed Acidity: 9.70
Sulphates: 0.74
Chlorides: 0.10

Characteristics:
highest acidity
highest sulphates
highest chlorides

Possible label: Highly structured acidic wines

- Most interesting finding

Cluster 2 is dramatically different:

Residual Sugar
Cluster 0 = 2.49
Cluster 1 = 3.26
Cluster 2 = 10.72  <- huge difference
Cluster 3 = 2.74

That tells me K-Means is finding a genuinely distinct wine profile rather than 
arbitrary groups.

Add this interpretation section to your script

At this point I would consider the K-Means section complete and move directly 
to Gaussian Mixture Models (GMM) for comparison. The K-Means chapter already 
contains enough evidence and interpretation for a professional portfolio project.
"""

=========================================================
Export Model Summary
=========================================================

In [ ]:
kmeans_summary = pd.DataFrame({
    "algorithm": ["K-Means"],
    "final_model": [f"k={best_k}"],
    "silhouette_score": [
        results_df.loc[
            results_df["k"] == best_k,
            "silhouette_score"
        ].iloc[0]
    ],
    "davies_bouldin_index": [
        results_df.loc[
            results_df["k"] == best_k,
            "davies_bouldin_index"
        ].iloc[0]
    ],
    "stability_ari": [np.mean(ari_scores)],
    "cluster_count": [best_k],
    "noise_pct": [0.0]
})

In [ ]:
kmeans_summary.to_csv(
    output_dir / "kmeans_summary.csv",
    index=False
)

=========================================================
K-Means: Key Insights
=========================================================

In [ ]:
"""
KEY INSIGHTS

• K-Means was evaluated across k values ranging from 2 to 20.

• The highest silhouette score (0.2704) occurred at k=2, but this solution
  was considered overly simplistic for meaningful segmentation.

• The lowest Davies-Bouldin Index (1.4844) occurred at k=4, indicating the
  best balance between cluster compactness and separation.

• The elbow plot showed diminishing reductions in inertia beyond four
  clusters, supporting k=4 as the optimal solution.

• Cluster stability was exceptionally high, achieving an average Adjusted
  Rand Index (ARI) of 0.9986 across multiple random seeds.

• PCA, t-SNE, and UMAP visualizations all revealed a clear four-cluster
  structure with limited overlap between groups.

• Cluster profiling identified distinct wine populations characterized by
  differences in alcohol content, residual sugar, acidity, sulphates, and
  sulfur dioxide concentrations.

• The final K-Means solution consisted of four stable and interpretable wine
  profiles.

FINAL MODEL

Algorithm: K-Means
Clusters: 4
Silhouette Score: 0.2428
Davies-Bouldin Index: 1.4844
Average ARI: 0.9986
"""